# Seeing what happens in the network

A `Cluster` wires a node class into the simulator. After a run, `messages()` lists every message and its fate, `diagram()` draws a space-time diagram, `explain(node)` shows one node's view, and `timeline()` lists every event. Each of these renders as a table or a picture when it is the last expression in a cell; `print(...)` gives the plain-text version.

In [ ]:
from dslabs import Cluster, drop, duplicate, partition
from dslabs.nodes import NodeMultiLeader

## One write, no faults

A client writes `x = 1` at n2. The naive multi-leader node applies it locally and broadcasts it. Time only advances when we say so.

In [ ]:
cluster = Cluster(NodeMultiLeader, 3, seed=1)
cluster.put("n2", "x", 1)
cluster.run_until(500)
cluster.values("x")

In [ ]:
cluster.messages()

In [ ]:
cluster.diagram()

## The same write with message loss

Half of all messages are dropped. The table names the rule that dropped each one, and `explain` answers the question a student will ask: why does n3 not have the value?

In [ ]:
cluster = Cluster(NodeMultiLeader, 3, seed=1)
cluster.add_rule(drop(0.5))
cluster.put("n2", "x", 1)
cluster.run_until(500)
cluster.values("x")

In [ ]:
cluster.messages()

In [ ]:
cluster.diagram()

In [ ]:
cluster.explain("n3")

## Reordering

Two writes 20 ms apart from different nodes. Latency varies per message, so a later write can arrive before an earlier one, and nodes end up disagreeing. Watch the arrows cross.

In [ ]:
cluster = Cluster(NodeMultiLeader, 3, seed=6, latency_ms=(30, 120))
cluster.put("n1", "x", 1)
cluster.run_until(20)
cluster.put("n3", "x", 2)
cluster.run_until(500)
cluster.values("x")

In [ ]:
cluster.diagram()

## Step by step: predict, then check

The same reordering run, one event at a time. `pending()` lists what is in flight, `peek()` says what happens next, and `step()` runs exactly one event and shows everything it caused. Before each step, say out loud what you expect.

In [ ]:
cluster = Cluster(NodeMultiLeader, 3, seed=6, latency_ms=(30, 120))
cluster.put("n1", "x", 1)
cluster.run_until(20)
cluster.put("n3", "x", 2)
cluster.pending()

Four messages are in flight. Which one lands first, and what will that node believe `x` is afterwards?

In [ ]:
cluster.peek()

In [ ]:
cluster.step()

n3 wrote `x = 2` itself at 20 ms, and has just overwritten it with the older write from n1. Nothing was lost or reordered on that link; n1's message was simply still in flight when n3 wrote.

n2 has heard nothing yet. It will receive both writes. Which arrives first, and which one will it end up with?

In [ ]:
cluster.step()

In [ ]:
cluster.step()

n2 received the newer write first and then the older one, so it ends with the stale value. One message left: n1 is about to hear about n3's write.

In [ ]:
cluster.step()

In [ ]:
cluster.step()   # nothing left to do

To run to the end without stopping, use `while cluster.step(): pass`, or `run_until_idle()`. The trace is the same either way, so the diagram of what we just stepped through is:

In [ ]:
cluster.diagram()

## A partition that heals

n1 is cut off for the first two seconds. Rules can be added and removed at any time, including from a timer.

In [ ]:
cluster = Cluster(NodeMultiLeader, 3, seed=2)
cut = partition({"n1"})
cluster.add_rule(cut)
cluster.scheduler.call_later(2000, lambda: cluster.remove_rule(cut))

cluster.put("n1", "x", 1)      # lost: n1 is isolated
cluster.run_until(2500)
cluster.put("n1", "x", 2)      # gets through
cluster.run_until(3000)
cluster.values("x")

In [ ]:
cluster.diagram()

In [ ]:
cluster.timeline()

## Every event, as text

The same information is available without a notebook: `print(cluster.messages())`, `print(cluster.timeline())`, `cluster.diagram().save("run.svg")`. Pass `verbose=True` to `Cluster` to see events printed as they happen.

In [ ]:
print(cluster.messages())

## Animated replay

For a longer run, export the trace and replay it in the animated viewer: nodes on a ring, messages as dots travelling between them, dropped copies turning into a cross partway, each node's state and pending timers underneath, and a clickable event log. This run has four nodes, four writes, and a network that both loses and duplicates messages.

In [ ]:
cluster = Cluster(NodeMultiLeader, 4, seed=3)
cluster.add_rule(drop(0.25))
cluster.add_rule(duplicate(0.2))
for t, node in [(0, "n1"), (150, "n3"), (300, "n2"), (450, "n4")]:
    cluster.run_until(t)
    cluster.put(node, "x", t // 150 + 1)
cluster.run_until_idle()
cluster.stats

In [ ]:
cluster.messages()

In [ ]:
cluster.save_json("run.json")      # for dslabs/viewer.html: open it in a browser and load this file
cluster.save_viewer("run.html")    # or one self-contained page: just open it

Open `run.html` in a browser. Space plays and pauses, the arrow keys step between events, the slider scrubs, and clicking an event in the log jumps to it. Add `#t=300` to the address to open at a particular moment. To replay any exported trace, open `dslabs/viewer.html` and drop the `.json` file onto it.